***ML100K with BPR, Reranking with LLAMA***

In [33]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
import random
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from llama_cpp import Llama

#################################
# BPR RECOMMENDER IMPLEMENTATION
#################################

class BPRRecommender:
    def __init__(self, factors=50, learning_rate=0.01, regularization=0.01, iterations=50, random_state=42):
        """
        Bayesian Personalized Ranking (BPR) recommender algorithm
        """
        self.factors = factors
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.iterations = iterations
        self.random_state = random_state
        np.random.seed(random_state)
        
    def fit(self, user_item_matrix):
        """
        Train the BPR model on the user-item matrix
        """
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        
        # Initialize latent factors
        self.user_factors = np.random.normal(0, 0.1, (self.n_users, self.factors))
        self.item_factors = np.random.normal(0, 0.1, (self.n_items, self.factors))
        
        # Create a dictionary of items each user has interacted with
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        
        # Training loop
        for iteration in range(self.iterations):
            # Sample triplets for training
            for _ in range(user_item_matrix.nnz):
                user, pos_item, neg_item = self._sample_triplet()
                self._update_factors(user, pos_item, neg_item)
            
            # Print progress
            if (iteration + 1) % 10 == 0:
                print(f"Completed iteration {iteration + 1}/{self.iterations}")
                
        return self
    
    def _sample_triplet(self):
        """
        Sample a (user, positive_item, negative_item) triplet for training
        """
        user = random.choice(list(self.user_items.keys()))
        pos_item = random.choice(list(self.user_items[user]))
        neg_item = random.randint(0, self.n_items - 1)
        while neg_item in self.user_items[user]:
            neg_item = random.randint(0, self.n_items - 1)
        return user, pos_item, neg_item
    
    def _update_factors(self, user, pos_item, neg_item):
        """
        Update model parameters based on a triplet
        """
        pos_pred = np.dot(self.user_factors[user], self.item_factors[pos_item])
        neg_pred = np.dot(self.user_factors[user], self.item_factors[neg_item])
        diff = neg_pred - pos_pred
        sigmoid = 1.0 / (1.0 + np.exp(-diff))
        
        grad_user = sigmoid * (self.item_factors[neg_item] - self.item_factors[pos_item]) + self.regularization * self.user_factors[user]
        grad_pos_item = sigmoid * (-self.user_factors[user]) + self.regularization * self.item_factors[pos_item]
        grad_neg_item = sigmoid * self.user_factors[user] + self.regularization * self.item_factors[neg_item]
        
        self.user_factors[user] -= self.learning_rate * grad_user
        self.item_factors[pos_item] -= self.learning_rate * grad_pos_item
        self.item_factors[neg_item] -= self.learning_rate * grad_neg_item
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        """
        Generate item recommendations for a user based on dot product scores.
        """
        scores = np.dot(self.user_factors[user_id], self.item_factors.T)
        if exclude_seen and user_id in self.user_items:
            seen_items = list(self.user_items[user_id])
            scores[seen_items] = -np.inf
        top_items = np.argsort(scores)[::-1][:n]
        return top_items

#################################
# LLM-BASED RERANKER IMPLEMENTATION für BPR
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Item-Popularität für Novelty/Fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

    def get_user_goal_from_llm(self, user_id, top_items):
        """
        Fragt das LLM basierend auf konkreten Items nach einem Ziel (accuracy, diverse_first, ...)
        """
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]
    
        # Baue Prompt
        lines = [f"{i+1}. {item['title']} (Genres: {', '.join(item['genres'])})"
                 for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)
        
        prompt = (
            f"<s>[INST] A recommender system suggests the following movies to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Choose one of the following goals for re-ranking:\n"
            f"- accuracy\n- diverse_first\n- fair_first\n- balance\n\n"
            f"Return only the goal word. [/INST]"
        )

    
        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()
    
        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                return goal
    
        # Fallback
        self.user_goal_map[user_id] = "balance"
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        """
        Generiert eine rerankte Empfehlungsliste für den Nutzer – Zielgewichtung durch LLM.
        """
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)
    
        # Hole Metadaten zu den empfohlenen Items
        top_items = []
        for item_idx in candidates[:10]:  # die Top 10 Kandidaten als Kontext
            original_id = self.reverse_item_mapping[item_idx]
            if original_id in self.item_metadata:
                top_items.append(self.item_metadata[original_id])
            else:
                top_items.append({'title': f"Item {original_id}", 'genres': []})
    
        # Prompt-Entscheidung via LLM
        goal = self.get_user_goal_from_llm(user_id, top_items)
    
        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))
    
        user_vector = self.model.user_factors[user_id]
        predicted_scores = np.dot(user_vector, self.model.item_factors.T)
    
        selected = []
        while len(selected) < n and candidates.size > 0:
            best_score = -np.inf
            best_item = None
            for item in candidates:
                if item in selected:
                    continue
    
                # Accuracy
                score_accuracy = predicted_scores[item]
    
                # Diversity
                if selected:
                    similarities = []
                    for sel_item in selected:
                        vec_item = self.model.item_factors[item]
                        vec_sel = self.model.item_factors[sel_item]
                        dot = np.dot(vec_item, vec_sel)
                        norm = np.linalg.norm(vec_item) * np.linalg.norm(vec_sel)
                        sim = dot / norm if norm > 0 else 0
                        similarities.append(sim)
                    avg_sim = np.mean(similarities)
                    diversity_score = 1 - avg_sim
                else:
                    diversity_score = 1
    
                # Fairness (Novelty)
                novelty_score = 1 - self.norm_popularity[item]
    
                # Gesamtbewertung
                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)
    
                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item
    
            if best_item is None:
                break
            selected.append(best_item)
            candidates = candidates[candidates != best_item]
    
        return selected



#################################
# EVALUATION METRICS (UNVERÄNDERT)
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return 0
    return dcg / idcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(recommended_items) if recommended_items else 0

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(relevant_items) if relevant_items else 0

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = sum((i + 1) * count for i, count in enumerate(sorted_counts))
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS: DATENLADEN UND MATRIXERSTELLUNG
#################################

def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def load_item_metadata(path="ml-100k/u.item"):
    columns = ['item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]
    genre_labels = [
        'Unknown', 'Action', 'Adventure', 'Animation', 'Children’s', 'Comedy', 'Crime', 'Documentary',
        'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]
    df = pd.read_csv(path, sep='|', encoding='latin-1', names=columns)

    item_info = {}
    for _, row in df.iterrows():
        genres = [genre_labels[i] for i in range(19) if row[f'genre_{i}'] == 1]
        item_info[row['item_id']] = {
            'title': row['title'],
            'genres': genres
        }
    return item_info


def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), 
                                  shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

#################################
# COMPREHENSIVE EVALUATION FÜR MEHRERE RERANKER
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading MovieLens 100K dataset...")
    ratings_df, movie_df = load_movielens_100k()
    
    print("Splitting data for evaluation...")
    train_df, test_df = train_test_split(
        ratings_df, 
        test_size=0.2, 
        stratify=ratings_df['user_id'], 
        random_state=42
    )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining BPR model...")
    model = BPRRecommender(factors=100, learning_rate=0.02, regularization=0.005, iterations=30)
    model.fit(user_item_matrix)
    
    # Initialisiere Rerankers: Original BPR und unser neuer LLM Reranker (SimpleReranker wurde entfernt)
    print("\nLoading local LLM model...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    item_metadata = load_item_metadata(path="ml-100k/u.item")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
    reverse_item_mapping=reverse_item_mapping
    )

    rerankers = {
        "Original BPR": None,
        "LLM Reranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:  # Original BPR
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    # Ausgabe der Ergebnisse
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original BPR"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original BPR":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original BPR"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original BPR":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures ratio of relevant items")
    print("- Recall: Higher is better, measures coverage of relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")
    
    return all_results

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading MovieLens 100K dataset...
Splitting data for evaluation...
Creating user-item matrix...

Training BPR model...
Completed iteration 10/30
Completed iteration 20/30
Completed iteration 30/30

Loading local LLM model...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Evaluating 943 users...

Evaluating Original BPR...

Evaluating LLM Reranker...


/home/stef/projects/myenv/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original BPR        LLM Reranker        
--------------------------------------------------------------------------------
ndcg@10        0.3056               0.2957 (-3.2%)     
precision@10   0.3308               0.3243 (-2.0%)     
recall@10      0.2119               0.2058 (-2.9%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original BPR        LLM Reranker        
--------------------------------------------------------------------------------
item_coverage  0.2627               0.2886 (+9.9%)     
gini_index     0.6848               0.6644 (-3.0%)     
shannon_entropy0.7031               0.7247 (+3.1%)     
tail_percentage0.0000               0.0000 (+inf%)     

============================== METRIC INTERPRETATIONS ==============================
Accuracy Metrics:
- NDCG: Higher is better, measures ranking qual

***ML100K with ItemKNN, Reranking with LLAMA***

In [29]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
import random
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

#################################
# ITEMKNN RECOMMENDER IMPLEMENTATION
#################################

class ItemKNNRecommender:
    def __init__(self, k=50, random_state=42):
        """
        Item-Based K-Nearest Neighbors recommender algorithm.
        
        Parameters:
        - k: number of nearest neighbors to consider
        - random_state: seed for reproducibility
        """
        self.k = k
        self.random_state = random_state
        np.random.seed(random_state)
        
    def fit(self, user_item_matrix):
        """
        Train the ItemKNN model on the user-item matrix.
        
        Parameters:
        - user_item_matrix: scipy sparse matrix with user-item interactions
        
        Returns:
        - self
        """
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        
        # Compute item-item similarity matrix
        print("Computing item-item similarity matrix...")
        self.item_factors = cosine_similarity(user_item_matrix.T)
        
        # Zero out self-similarity to avoid recommending the same item
        np.fill_diagonal(self.item_factors, 0)
        
        # Create a dictionary of items each user has interacted with
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        
        # For each item, find its k nearest neighbors
        self.item_neighbors = {}
        for item_id in range(self.n_items):
            similarities = self.item_factors[item_id]
            neighbor_ids = np.argsort(similarities)[::-1][:self.k]
            self.item_neighbors[item_id] = {
                neighbor_id: similarities[neighbor_id] 
                for neighbor_id in neighbor_ids 
                if similarities[neighbor_id] > 0
            }
            
        return self
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        """
        Generate item recommendations for a user.
        
        Parameters:
        - user_id: user index
        - n: number of recommendations to generate
        - exclude_seen: whether to exclude items the user has already interacted with
        
        Returns:
        - list of n recommended item indices
        """
        if user_id not in self.user_items or not self.user_items[user_id]:
            all_items = set(range(self.n_items))
            seen_items = self.user_items.get(user_id, set())
            candidate_items = list(all_items - seen_items) if exclude_seen else list(all_items)
            if len(candidate_items) <= n:
                return candidate_items
            return random.sample(candidate_items, n)
        
        scores = np.zeros(self.n_items)
        for item_id in self.user_items[user_id]:
            if item_id in self.item_neighbors:
                for neighbor_id, similarity in self.item_neighbors[item_id].items():
                    scores[neighbor_id] += similarity
        
        if exclude_seen:
            for item_id in self.user_items[user_id]:
                scores[item_id] = -np.inf
                
        top_items = np.argsort(scores)[::-1][:n]
        return top_items

#################################
# LLM-BASED RERANKER IMPLEMENTATION
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Item-Popularität berechnen für Novelty / Fairness
        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]

        lines = [f"{i+1}. {item['title']} (Genres: {', '.join(item['genres'])})"
                 for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)

        prompt = (
            f"<s>[INST] A recommender system suggests the following movies to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Choose one of the following goals for re-ranking:\n"
            f"- accuracy\n- diverse_first\n- fair_first\n- balance\n\n"
            f"Return only the goal word. [/INST]"
        )

        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()

        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                return goal

        self.user_goal_map[user_id] = "balance"
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Hole Metadaten zu den empfohlenen Items
        top_items = []
        for item_idx in candidates[:10]:
            original_id = self.reverse_item_mapping[item_idx]
            if original_id in self.item_metadata:
                top_items.append(self.item_metadata[original_id])
            else:
                top_items.append({'title': f"Item {original_id}", 'genres': []})

        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        # Accuracy-Scores vorab berechnen
        scores = np.zeros(self.model.n_items)
        for item in self.model.user_items.get(user_id, []):
            if item in self.model.item_neighbors:
                for neighbor, similarity in self.model.item_neighbors[item].items():
                    scores[neighbor] += similarity

        selected = []
        while len(selected) < n and len(candidates) > 0:
            best_score = -np.inf
            best_item = None

            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = scores[item]

                if selected:
                    similarities = [self.model.item_factors[item, sel_item] for sel_item in selected]
                    avg_similarity = np.mean(similarities) if similarities else 0
                    diversity_score = 1 - avg_similarity
                else:
                    diversity_score = 1

                novelty_score = 1 - self.norm_popularity[item]

                combined_score = (w1 * score_accuracy +
                                  w2 * diversity_score +
                                  w3 * novelty_score)

                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item

            if best_item is None:
                break

            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return selected


#################################
# EVALUATION METRICS (UNVERÄNDERT)
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return 0
    ndcg = dcg / idcg
    return ndcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    precision = num_relevant_recommended / len(recommended_items) if recommended_items else 0
    return precision

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    recall = num_relevant_recommended / len(relevant_items) if relevant_items else 0
    return recall

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = sum((i + 1) * count for i, count in enumerate(sorted_counts))
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS (DATENLADEN)
#################################

def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def load_item_metadata(path="ml-100k/u.item"):
    columns = ['item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]
    genre_labels = [
        'Unknown', 'Action', 'Adventure', 'Animation', 'Children’s', 'Comedy', 'Crime', 'Documentary',
        'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]
    df = pd.read_csv(path, sep='|', encoding='latin-1', names=columns)

    item_info = {}
    for _, row in df.iterrows():
        genres = [genre_labels[i] for i in range(19) if row[f'genre_{i}'] == 1]
        item_info[row['item_id']] = {
            'title': row['title'],
            'genres': genres
        }
    return item_info

def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading MovieLens 100K dataset...")
    ratings_df, movie_df = load_movielens_100k()
    
    print("Splitting data for evaluation...")
    train_df, test_df = train_test_split(
        ratings_df, 
        test_size=0.2, 
        stratify=ratings_df['user_id'], 
        random_state=42
    )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining ItemKNN model...")
    model = ItemKNNRecommender(k=50)
    model.fit(user_item_matrix)
    
    print("\nInitializing reranker (LLM-based)...")
    from llama_cpp import Llama

    # LLM initialisieren
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
    # Item-Metadaten laden
    item_metadata = load_item_metadata(path="ml-100k/u.item")
    
    # LLM-Reranker initialisieren
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    
    rerankers = {
        "Original ItemKNN": None,
        "ItemKNN + LLMReranker": llm_reranker
    }
    
    all_results = {}
    
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            
            ndcg_scores.append(calculate_ndcg(
                rec, test_relevant_items[user_id], test_relevant_scores[user_id]
            ))
            precision_scores.append(calculate_precision(
                rec, test_relevant_items[user_id]
            ))
            recall_scores.append(calculate_recall(
                rec, test_relevant_items[user_id]
            ))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original ItemKNN"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original ItemKNN":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original ItemKNN"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original ItemKNN":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of all relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in item recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")
    
    return all_results

# Execute the evaluation when running the script directly
if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading MovieLens 100K dataset...
Splitting data for evaluation...
Creating user-item matrix...

Training ItemKNN model...
Computing item-item similarity matrix...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Initializing reranker (LLM-based)...

Evaluating 943 users...

Evaluating Original ItemKNN...

Evaluating ItemKNN + LLMReranker...


/home/stef/projects/myenv/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original ItemKNN    ItemKNN + LLMReranker
--------------------------------------------------------------------------------
ndcg@10        0.2763               0.2758 (-0.2%)     
precision@10   0.2935               0.2936 (+0.0%)     
recall@10      0.1928               0.1932 (+0.2%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original ItemKNN    ItemKNN + LLMReranker
--------------------------------------------------------------------------------
item_coverage  0.1165               0.1214 (+4.1%)     
gini_index     0.6928               0.6983 (+0.8%)     
shannon_entropy0.5913               0.5944 (+0.5%)     
tail_percentage0.0008               0.0012 (+37.5%)     

============================== METRIC INTERPRETATIONS ==============================
Accuracy Metrics:
- NDCG: Higher is better, measures ranking q

***ML100K with NeuMF, Reranking with LLAMA***

In [30]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
import random
import time
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from llama_cpp import Llama

#################################
# NEUMF RECOMMENDER IMPLEMENTATION
#################################

class NCFDataset(Dataset):
    """Dataset for NCF"""
    def __init__(self, user_item_matrix, neg_samples=4):
        self.user_item_matrix = user_item_matrix
        self.users, self.items = user_item_matrix.nonzero()
        self.n_users = user_item_matrix.shape[0]
        self.n_items = user_item_matrix.shape[1]
        self.neg_samples = neg_samples
        self.user_item_set = set(zip(self.users, self.items))
        self.user_items = defaultdict(set)
        for u, i in zip(self.users, self.items):
            self.user_items[u].add(i)
    
    def __len__(self):
        return len(self.users) * (1 + self.neg_samples)
    
    def __getitem__(self, idx):
        if idx < len(self.users):
            user = self.users[idx]
            item = self.items[idx]
            label = 1.0
        else:
            pos_idx = idx % len(self.users)
            user = self.users[pos_idx]
            item = random.randint(0, self.n_items - 1)
            while item in self.user_items[user]:
                item = random.randint(0, self.n_items - 1)
            label = 0.0
        return user, item, label

class GMF(nn.Module):
    """Generalized Matrix Factorization model"""
    def __init__(self, n_users, n_items, latent_dim):
        super(GMF, self).__init__()
        self.user_embedding = nn.Embedding(n_users, latent_dim)
        self.item_embedding = nn.Embedding(n_items, latent_dim)
        self.output_layer = nn.Linear(latent_dim, 1)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        
    def forward(self, user_indices, item_indices):
        user_embeddings = self.user_embedding(user_indices)
        item_embeddings = self.item_embedding(item_indices)
        element_product = torch.mul(user_embeddings, item_embeddings)
        output = self.output_layer(element_product)
        return output.view(-1)

class MLP(nn.Module):
    """Multi-Layer Perceptron model"""
    def __init__(self, n_users, n_items, latent_dim, layers=[64, 32, 16, 8]):
        super(MLP, self).__init__()
        self.user_embedding = nn.Embedding(n_users, latent_dim)
        self.item_embedding = nn.Embedding(n_items, latent_dim)
        self.layers = nn.ModuleList()
        layer_dims = [2 * latent_dim] + layers
        for i in range(len(layer_dims) - 1):
            self.layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            self.layers.append(nn.ReLU())
        self.output_layer = nn.Linear(layer_dims[-1], 1)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        
    def forward(self, user_indices, item_indices):
        user_embeddings = self.user_embedding(user_indices)
        item_embeddings = self.item_embedding(item_indices)
        vector = torch.cat([user_embeddings, item_embeddings], dim=-1)
        for layer in self.layers:
            vector = layer(vector)
        output = self.output_layer(vector)
        return output.view(-1)

class NeuMF(nn.Module):
    """Neural Matrix Factorization model"""
    def __init__(self, n_users, n_items, latent_dim=32, mlp_layers=[64, 32, 16, 8]):
        super(NeuMF, self).__init__()
        self.gmf = GMF(n_users, n_items, latent_dim)
        self.mlp = MLP(n_users, n_items, latent_dim, mlp_layers)
        self.output_layer = nn.Linear(mlp_layers[-1] + latent_dim, 1)
        nn.init.normal_(self.output_layer.weight, std=0.01)
        
    def forward(self, user_indices, item_indices):
        # GMF path
        gmf_user = self.gmf.user_embedding(user_indices)
        gmf_item = self.gmf.item_embedding(item_indices)
        gmf_vector = torch.mul(gmf_user, gmf_item)
        # MLP path
        mlp_user = self.mlp.user_embedding(user_indices)
        mlp_item = self.mlp.item_embedding(item_indices)
        mlp_vector = torch.cat([mlp_user, mlp_item], dim=-1)
        for layer in self.mlp.layers:
            mlp_vector = layer(mlp_vector)
        vector = torch.cat([gmf_vector, mlp_vector], dim=-1)
        output = self.output_layer(vector)
        return torch.sigmoid(output.view(-1))

class NeuMFRecommender:
    def __init__(self, latent_dim=32, mlp_layers=[64, 32, 16, 8], epochs=20, batch_size=256, 
                 lr=0.001, neg_samples=4, device=None, random_state=42):
        self.latent_dim = latent_dim
        self.mlp_layers = mlp_layers
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.neg_samples = neg_samples
        self.random_state = random_state
        random.seed(random_state)
        np.random.seed(random_state)
        torch.manual_seed(random_state)
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device
        
    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        dataset = NCFDataset(user_item_matrix, neg_samples=self.neg_samples)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        self.model = NeuMF(self.n_users, self.n_items, self.latent_dim, self.mlp_layers).to(self.device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        print(f"Training NeuMF model for {self.epochs} epochs...")
        self.model.train()
        for epoch in range(self.epochs):
            start_time = time.time()
            running_loss = 0.0
            for users, items, labels in dataloader:
                users = users.to(self.device)
                items = items.to(self.device)
                labels = labels.float().to(self.device)
                outputs = self.model(users, items)
                loss = criterion(outputs, labels)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            elapsed_time = time.time() - start_time
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{self.epochs}, Loss: {running_loss/len(dataloader):.4f}, Time: {elapsed_time:.2f}s")
        self.model.eval()
        with torch.no_grad():
            self.user_gmf_embeddings = self.model.gmf.user_embedding.weight.data
            self.item_gmf_embeddings = self.model.gmf.item_embedding.weight.data
            self.user_mlp_embeddings = self.model.mlp.user_embedding.weight.data
            self.item_mlp_embeddings = self.model.mlp.item_embedding.weight.data
        # Kombiniere GMF- und MLP-Itemfaktoren
        self.item_factors = np.concatenate([
            self.item_gmf_embeddings.cpu().numpy(),
            self.item_mlp_embeddings.cpu().numpy()
        ], axis=1)
        return self
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        if user_id not in self.user_items:
            all_items = list(range(self.n_items))
            recommendations = random.sample(all_items, min(n, len(all_items)))
            return np.array(recommendations)
        with torch.no_grad():
            user_tensor = torch.LongTensor([user_id] * self.n_items).to(self.device)
            item_tensor = torch.LongTensor(list(range(self.n_items))).to(self.device)
            scores = self.model(user_tensor, item_tensor).cpu().numpy()
        if exclude_seen:
            for item_id in self.user_items[user_id]:
                scores[item_id] = -np.inf
        top_items = np.argsort(scores)[::-1][:n]
        return top_items

#################################
# LLM-BASED RERANKER IMPLEMENTATION für NeuMF
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        self.item_popularity = np.zeros(model.n_items)
        for user in model.user_items:
            for item in model.user_items[user]:
                self.item_popularity[item] += 1
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]
        lines = [f"{i+1}. {item['title']} (Genres: {', '.join(item['genres'])})" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)
        prompt = (
            f"<s>[INST] A recommender system suggests the following movies to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Choose one of the following goals for re-ranking:\n"
            f"- accuracy\n- diverse_first\n- fair_first\n- balance\n\n"
            f"Return only the goal word. [/INST]"
        )
        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()
        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                return goal
        self.user_goal_map[user_id] = "balance"
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Metadaten für Prompt
        top_items = []
        for item_idx in candidates[:10]:
            original_id = self.reverse_item_mapping[item_idx]
            if original_id in self.item_metadata:
                top_items.append(self.item_metadata[original_id])
            else:
                top_items.append({'title': f"Item {original_id}", 'genres': []})

        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        # Vorhersage-Scores vorbereiten
        with torch.no_grad():
            user_tensor = torch.LongTensor([user_id] * self.model.n_items).to(self.model.device)
            item_tensor = torch.LongTensor(list(range(self.model.n_items))).to(self.model.device)
            scores = self.model.model(user_tensor, item_tensor).cpu().numpy()

        selected = []
        while len(selected) < n and candidates.size > 0:
            best_score = -np.inf
            best_item = None
            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = scores[item]
                if selected:
                    similarities = []
                    for sel_item in selected:
                        vec_item = self.model.item_factors[item]
                        vec_sel = self.model.item_factors[sel_item]
                        dot = np.dot(vec_item, vec_sel)
                        norm = np.linalg.norm(vec_item) * np.linalg.norm(vec_sel)
                        sim = dot / norm if norm > 0 else 0
                        similarities.append(sim)
                    avg_sim = np.mean(similarities)
                    diversity_score = 1 - avg_sim
                else:
                    diversity_score = 1
                novelty_score = 1 - self.norm_popularity[item]
                combined_score = w1 * score_accuracy + w2 * diversity_score + w3 * novelty_score
                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item
            if best_item is None:
                break
            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return np.array(selected)


#################################
# EVALUATION METRICS (UNVERÄNDERT)
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return 0
    return dcg / idcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(recommended_items) if recommended_items else 0

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(relevant_items) if relevant_items else 0

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = sum((i + 1) * count for i, count in enumerate(sorted_counts))
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS: DATENLADEN UND MATRIXERSTELLUNG
#################################

def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("="*80)
    
    print("\nLoading MovieLens 100K dataset...")
    ratings_df, movie_df = load_movielens_100k()
    
    print("Splitting data for evaluation...")
    train_df, test_df = train_test_split(
        ratings_df, test_size=0.2, stratify=ratings_df['user_id'], random_state=42
    )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining NeuMF model...")
    model = NeuMFRecommender(latent_dim=64, epochs=10, batch_size=128, lr=0.001)
    model.fit(user_item_matrix)

    from llama_cpp import Llama
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
    def load_item_metadata(path="ml-100k/u.item"):
        columns = ['item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]
        genre_labels = [
            'Unknown', 'Action', 'Adventure', 'Animation', 'Children’s', 'Comedy', 'Crime', 'Documentary',
            'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
            'Thriller', 'War', 'Western'
        ]
        df = pd.read_csv(path, sep='|', encoding='latin-1', names=columns)
        item_info = {}
        for _, row in df.iterrows():
            genres = [genre_labels[i] for i in range(19) if row[f'genre_{i}'] == 1]
            item_info[row['item_id']] = {
                'title': row['title'],
                'genres': genres
            }
        return item_info
    
    item_metadata = load_item_metadata()

    
    # Initialisiere Rerankers: Es werden nur Original NeuMF und unser neuer LLM Reranker verwendet.
    print("\nInitializing rerankers...")
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )
    
    rerankers = {
        "Original NeuMF": None,
        "LLM Reranker": llm_reranker
    }

    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:  # Original NeuMF
                rec_idx = model.recommend(user_idx, n=k)
            else:  # LLM Reranker
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "="*30 + " ACCURACY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original NeuMF"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original NeuMF":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " DIVERSITY METRICS COMPARISON " + "="*30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original NeuMF"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original NeuMF":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "="*30 + " METRIC INTERPRETATIONS " + "="*30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of relevant items")
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")
    
    return all_results

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading MovieLens 100K dataset...
Splitting data for evaluation...
Creating user-item matrix...

Training NeuMF model...
Training NeuMF model for 10 epochs...
Epoch 1/10, Loss: 0.3813, Time: 4.27s
Epoch 5/10, Loss: 0.2489, Time: 4.10s


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Epoch 10/10, Loss: 0.1978, Time: 3.55s

Initializing rerankers...

Evaluating 943 users...

Evaluating Original NeuMF...

Evaluating LLM Reranker...


/home/stef/projects/myenv/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original NeuMF      LLM Reranker        
--------------------------------------------------------------------------------
ndcg@10        0.2657               0.1451 (-45.4%)     
precision@10   0.2928               0.1940 (-33.8%)     
recall@10      0.1942               0.1203 (-38.0%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original NeuMF      LLM Reranker        
--------------------------------------------------------------------------------
item_coverage  0.4444               0.5465 (+23.0%)     
gini_index     0.6376               0.5407 (-15.2%)     
shannon_entropy0.7905               0.8516 (+7.7%)     
tail_percentage0.0000               0.0000 (+inf%)     

============================== METRIC INTERPRETATIONS ==============================
Accuracy Metrics:
- NDCG: Higher is better, measures ranking

***ML100K with Pop, Reranking with LLAMA***

In [31]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
import random
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama

#################################
# POPULARITY RECOMMENDER IMPLEMENTATION
#################################

class PopRecommender:
    def __init__(self, random_state=42):
        """
        Popularity-based recommender algorithm
        
        Parameters:
        - random_state: seed for reproducibility
        """
        self.random_state = random_state
        random.seed(random_state)
        np.random.seed(random_state)
        
    def fit(self, user_item_matrix):
        """
        Compute global item popularity from the training data
        
        Parameters:
        - user_item_matrix: scipy sparse matrix with user-item interactions
        
        Returns:
        - self
        """
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        
        # Create a dictionary of items each user has interacted with
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        
        # Calculate item popularity (number of interactions per item)
        self.item_popularity = np.array(user_item_matrix.sum(axis=0)).flatten()
        
        # Create item factors für Kompatibilität mit Rerankern.
        # Hier verwenden wir 32 Dimensionen: in der ersten Dimension wird die normalisierte Popularität abgelegt.
        self.item_factors = np.zeros((self.n_items, 32))
        max_pop = np.max(self.item_popularity)
        if max_pop > 0:
            self.item_factors[:, 0] = self.item_popularity / max_pop
        
        # Die restlichen Dimensionen füllen wir mit zufälliger Rauschen, beeinflusst von der Popularität
        for i in range(self.n_items):
            np.random.seed(self.random_state + i)
            self.item_factors[i, 1:] = np.random.normal(0, 0.1, 31) * (0.5 + 0.5 * self.item_factors[i, 0])
        
        print("Popularity-based recommender ready! Top 5 most popular items:", 
              np.argsort(self.item_popularity)[::-1][:5])
        return self
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        """
        Generate item recommendations for a user based on global popularity
        
        Parameters:
        - user_id: user index
        - n: number of recommendations to generate
        - exclude_seen: whether to exclude items the user has already interacted with
        
        Returns:
        - list of n recommended item indices
        """
        # Beginne mit allen Items, sortiert nach absteigender Popularität.
        recommended_items = np.argsort(self.item_popularity)[::-1]
        
        # Falls nötig: items, die der Nutzer bereits gesehen hat, entfernen.
        if exclude_seen and user_id in self.user_items:
            seen_items = list(self.user_items[user_id])
            recommended_items = np.array([item for item in recommended_items if item not in seen_items])
        
        return recommended_items[:n]


#################################
# LLM-BASED RERANKER IMPLEMENTATION für PopRecommender
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}

        # Normierte Popularität
        self.item_popularity = self.model.item_popularity
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]
        lines = [f"{i+1}. {item['title']} (Genres: {', '.join(item['genres'])})" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)
        prompt = (
            f"<s>[INST] A recommender system suggests the following movies to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Choose one of the following goals for re-ranking:\n"
            f"- accuracy\n- diverse_first\n- fair_first\n- balance\n\n"
            f"Return only the goal word. [/INST]"
        )
        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()
        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                return goal
        self.user_goal_map[user_id] = "balance"
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)

        # Metadaten für Prompt
        top_items = []
        for item_idx in candidates[:10]:
            original_id = self.reverse_item_mapping[item_idx]
            if original_id in self.item_metadata:
                top_items.append(self.item_metadata[original_id])
            else:
                top_items.append({'title': f"Item {original_id}", 'genres': []})

        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        selected = []
        while len(selected) < n and candidates.size > 0:
            best_score = -np.inf
            best_item = None
            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = self.norm_popularity[item]
                if selected:
                    similarities = []
                    for sel_item in selected:
                        item_factors = self.model.item_factors[item]
                        sel_factors = self.model.item_factors[sel_item]
                        dot = np.dot(item_factors, sel_factors)
                        norm = np.linalg.norm(item_factors) * np.linalg.norm(sel_factors)
                        sim = dot / norm if norm > 0 else 0
                        similarities.append(sim)
                    avg_sim = np.mean(similarities) if similarities else 0
                    diversity_score = 1 - avg_sim
                else:
                    diversity_score = 1
                novelty_score = 1 - self.norm_popularity[item]
                combined_score = w1 * score_accuracy + w2 * diversity_score + w3 * novelty_score
                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item
            if best_item is None:
                break
            selected.append(best_item)
            candidates = candidates[candidates != best_item]

        return np.array(selected)

#################################
# EVALUATION METRICS (UNVERÄNDERT)
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return 0
    return dcg / idcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(recommended_items) if recommended_items else 0

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(relevant_items) if relevant_items else 0

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = sum((i + 1) * count for i, count in enumerate(sorted_counts))
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS: DATENLADEN UND MATRIXERSTELLUNG
#################################

def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def load_item_metadata(path="ml-100k/u.item"):
    columns = ['item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]
    genre_labels = [
        'Unknown', 'Action', 'Adventure', 'Animation', 'Children’s', 'Comedy', 'Crime', 'Documentary',
        'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]
    df = pd.read_csv(path, sep='|', encoding='latin-1', names=columns)
    item_info = {}
    for _, row in df.iterrows():
        genres = [genre_labels[i] for i in range(19) if row[f'genre_{i}'] == 1]
        item_info[row['item_id']] = {
            'title': row['title'],
            'genres': genres
        }
    return item_info


def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("=" * 80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("=" * 80)
    
    print("\nLoading MovieLens 100K dataset...")
    ratings_df, movie_df = load_movielens_100k()
    
    print("Splitting data for evaluation...")
    train_df, test_df = train_test_split(
        ratings_df, test_size=0.2, stratify=ratings_df['user_id'], random_state=42
    )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nTraining Popularity-based recommender...")
    model = PopRecommender()
    model.fit(user_item_matrix)
    
    # Initialisiere Rerankers: Es werden nur Original PopRecommender und unser neuer LLM Reranker verwendet.
    print("\nInitializing rerankers...")
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
    item_metadata = load_item_metadata()
    
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )

    rerankers = {
        "Original Pop": None,
        "LLM Reranker": llm_reranker
    }
    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:  # Original PopRecommender
                rec_idx = model.recommend(user_idx, n=k)
            else:
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "=" * 30 + " ACCURACY METRICS COMPARISON " + "=" * 30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Pop"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Pop":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "=" * 30 + " DIVERSITY METRICS COMPARISON " + "=" * 30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Pop"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Pop":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "=" * 30 + " METRIC INTERPRETATIONS " + "=" * 30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of relevant items")
    
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")
    
    return all_results

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading MovieLens 100K dataset...
Splitting data for evaluation...
Creating user-item matrix...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Training Popularity-based recommender...
Popularity-based recommender ready! Top 5 most popular items: [108  88  84 255 390]

Initializing rerankers...

Evaluating 943 users...

Evaluating Original Pop...

Evaluating LLM Reranker...


/home/stef/projects/myenv/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original Pop        LLM Reranker        
--------------------------------------------------------------------------------
ndcg@10        0.1687               0.1625 (-3.7%)     
precision@10   0.1906               0.1841 (-3.4%)     
recall@10      0.1184               0.1104 (-6.7%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original Pop        LLM Reranker        
--------------------------------------------------------------------------------
item_coverage  0.0302               0.0465 (+54.0%)     
gini_index     0.6275               0.7054 (+12.4%)     
shannon_entropy0.4268               0.4577 (+7.2%)     
tail_percentage0.0000               0.0000 (+inf%)     

============================== METRIC INTERPRETATIONS ==============================
Accuracy Metrics:
- NDCG: Higher is better, measures ranking qu

***ML100K with Random, Reranking with LLAMA***

In [32]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import math
import random
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from llama_cpp import Llama

#################################
# RANDOM RECOMMENDER IMPLEMENTATION
#################################

class RandomRecommender:
    def __init__(self, random_state=42):
        """
        Random recommender algorithm
        
        Parameters:
        - random_state: seed for reproducibility
        """
        self.random_state = random_state
        # Set the random seed for reproducibility
        random.seed(self.random_state)
        np.random.seed(self.random_state)
        
    def fit(self, user_item_matrix):
        """
        Store basic dataset information
        
        Parameters:
        - user_item_matrix: scipy sparse matrix with user-item interactions
        
        Returns:
        - self
        """
        self.user_item_matrix = user_item_matrix
        self.n_users, self.n_items = user_item_matrix.shape
        
        # Create a dictionary of items each user has interacted with
        self.user_items = defaultdict(set)
        for user, item in zip(*self.user_item_matrix.nonzero()):
            self.user_items[user].add(item)
        
        # For compatibility with rerankers, create dummy item factors.
        # These werden für Ähnlichkeitsberechnungen genutzt.
        np.random.seed(self.random_state)
        self.item_factors = np.random.normal(0, 0.1, (self.n_items, 32))
        
        # Create a dummy item popularity array: Alle Items werden als gleich beliebt angenommen.
        self.item_popularity = np.ones(self.n_items)
        print(f"Random recommender ready! Total items: {self.n_items}")
        return self
    
    def recommend(self, user_id, n=10, exclude_seen=True):
        """
        Generate random recommendations for a user
        
        Parameters:
        - user_id: user index
        - n: number of recommendations to generate
        - exclude_seen: whether to exclude items the user has already interacted with
        
        Returns:
        - np.array of n randomly recommended item indices
        """
        local_random = random.Random(self.random_state + user_id)
        all_items = list(range(self.n_items))
        if exclude_seen and user_id in self.user_items:
            candidate_items = [item for item in all_items if item not in self.user_items[user_id]]
        else:
            candidate_items = all_items
        
        if len(candidate_items) <= n:
            return np.array(candidate_items)
        recommended_items = local_random.sample(candidate_items, n)
        return np.array(recommended_items)

#################################
# LLM-BASED RERANKER IMPLEMENTATION für Random Recommender
#################################

class LLMReranker:
    def __init__(self, model, llm, item_metadata, reverse_item_mapping):
        self.model = model
        self.llm = llm
        self.item_metadata = item_metadata
        self.reverse_item_mapping = reverse_item_mapping
        self.user_goal_map = {}
        
        self.item_popularity = self.model.item_popularity
        max_pop = np.max(self.item_popularity)
        self.norm_popularity = self.item_popularity / max_pop if max_pop > 0 else np.zeros_like(self.item_popularity)

    def get_user_goal_from_llm(self, user_id, top_items):
        if user_id in self.user_goal_map:
            return self.user_goal_map[user_id]
        lines = [f"{i+1}. {item['title']} (Genres: {', '.join(item['genres'])})" for i, item in enumerate(top_items)]
        item_descriptions = "\n".join(lines)
        prompt = (
            f"<s>[INST] A recommender system suggests the following movies to user {user_id}:\n"
            f"{item_descriptions}\n\n"
            f"Choose one of the following goals for re-ranking:\n"
            f"- accuracy\n- diverse_first\n- fair_first\n- balance\n\n"
            f"Return only the goal word. [/INST]"
        )
        response = self.llm(prompt, max_tokens=20, stop=["</s>"])
        text = response["choices"][0]["text"].lower()
        for goal in ["accuracy", "diverse_first", "fair_first", "balance"]:
            if goal in text:
                self.user_goal_map[user_id] = goal
                return goal
        self.user_goal_map[user_id] = "balance"
        return "balance"

    def rerank(self, user_id, n=10, candidate_size=30):
        candidates = self.model.recommend(user_id, n=candidate_size, exclude_seen=True)
        candidates = np.array(candidates)
        np.random.seed(self.model.random_state + user_id)
        random_scores = np.random.random(self.model.n_items)

        # Metadaten für Prompt
        top_items = []
        for item_idx in candidates[:10]:
            original_id = self.reverse_item_mapping[item_idx]
            if original_id in self.item_metadata:
                top_items.append(self.item_metadata[original_id])
            else:
                top_items.append({'title': f"Item {original_id}", 'genres': []})

        goal = self.get_user_goal_from_llm(user_id, top_items)

        weight_map = {
            "accuracy": (0.7, 0.15, 0.15),
            "diverse_first": (0.4, 0.4, 0.2),
            "fair_first": (0.4, 0.2, 0.4),
            "balance": (0.5, 0.25, 0.25)
        }
        w1, w2, w3 = weight_map.get(goal, (0.5, 0.25, 0.25))

        selected = []
        while len(selected) < n and candidates.size > 0:
            best_score = -np.inf
            best_item = None
            for item in candidates:
                if item in selected:
                    continue
                score_accuracy = random_scores[item]
                if selected:
                    similarities = []
                    for sel_item in selected:
                        vec_item = self.model.item_factors[item]
                        vec_sel = self.model.item_factors[sel_item]
                        dot = np.dot(vec_item, vec_sel)
                        norm = np.linalg.norm(vec_item) * np.linalg.norm(vec_sel)
                        sim = dot / norm if norm > 0 else 0
                        similarities.append(sim)
                    avg_sim = np.mean(similarities)
                    diversity_score = 1 - avg_sim
                else:
                    diversity_score = 1
                novelty_score = 1 - self.norm_popularity[item]
                combined_score = w1 * score_accuracy + w2 * diversity_score + w3 * novelty_score
                if combined_score > best_score:
                    best_score = combined_score
                    best_item = item
            if best_item is None:
                break
            selected.append(best_item)
            candidates = candidates[candidates != best_item]
        return np.array(selected)

#################################
# EVALUATION METRICS (UNVERÄNDERT)
#################################

def calculate_ndcg(recommended_items, relevant_items, relevant_scores, k=None):
    if k is None:
        k = len(recommended_items)
    else:
        k = min(k, len(recommended_items))
    relevance_map = {item_id: score for item_id, score in zip(relevant_items, relevant_scores)}
    dcg = 0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevance_map:
            rel = relevance_map[item_id]
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    sorted_relevant = sorted(zip(relevant_items, relevant_scores), key=lambda x: x[1], reverse=True)
    idcg = 0
    for i, (item_id, rel) in enumerate(sorted_relevant[:k]):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return 0
    return dcg / idcg

def calculate_precision(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(recommended_items) if recommended_items else 0

def calculate_recall(recommended_items, relevant_items):
    num_relevant_recommended = sum(1 for item in recommended_items if item in relevant_items)
    return num_relevant_recommended / len(relevant_items) if relevant_items else 0

def calculate_diversity_metrics(recommendations, item_popularity, total_items, tail_items=None):
    rec_counts = Counter(recommendations)
    recommended_items = len(rec_counts)
    item_coverage = recommended_items / total_items
    sorted_counts = sorted(rec_counts.values())
    n = len(sorted_counts)
    if n == 0:
        gini_index = 0
    else:
        cumulative_sum = sum((i + 1) * count for i, count in enumerate(sorted_counts))
        gini_index = (2 * cumulative_sum) / (n * sum(sorted_counts)) - (n + 1) / n
    recommendations_count = sum(rec_counts.values())
    probabilities = [count / recommendations_count for count in rec_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    max_entropy = np.log2(min(total_items, recommendations_count))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    if tail_items is None:
        sorted_pop_indices = np.argsort(item_popularity)
        num_tail_items = int(len(sorted_pop_indices) * 0.2)
        tail_items = set(sorted_pop_indices[:num_tail_items])
    tail_recommendations = sum(1 for item in recommendations if item in tail_items)
    tail_percentage = tail_recommendations / len(recommendations) if recommendations else 0
    metrics = {
        'item_coverage': item_coverage,
        'gini_index': gini_index,
        'shannon_entropy': normalized_entropy,
        'tail_percentage': tail_percentage
    }
    return metrics, tail_items

#################################
# HELPER FUNCTIONS: DATENLADEN UND MATRIXERSTELLUNG
#################################

def load_movielens_100k(path="ml-100k"):
    ratings_df = pd.read_csv(f"{path}/u.data", sep='\t', 
                             names=['user_id', 'item_id', 'rating', 'timestamp'])
    movie_df = pd.read_csv(f"{path}/u.item", sep='|', encoding='latin-1',
                           names=['item_id', 'title', 'release_date', 'video_release_date',
                                  'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
    return ratings_df, movie_df

def load_item_metadata(path="ml-100k/u.item"):
    columns = ['item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]
    genre_labels = [
        'Unknown', 'Action', 'Adventure', 'Animation', 'Children’s', 'Comedy', 'Crime', 'Documentary',
        'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]
    df = pd.read_csv(path, sep='|', encoding='latin-1', names=columns)
    item_info = {}
    for _, row in df.iterrows():
        genres = [genre_labels[i] for i in range(19) if row[f'genre_{i}'] == 1]
        item_info[row['item_id']] = {
            'title': row['title'],
            'genres': genres
        }
    return item_info


def create_user_item_matrix(ratings_df):
    user_ids = ratings_df['user_id'].unique()
    item_ids = ratings_df['item_id'].unique()
    user_mapping = {user_id: i for i, user_id in enumerate(user_ids)}
    item_mapping = {item_id: i for i, item_id in enumerate(item_ids)}
    rows = ratings_df['user_id'].map(user_mapping)
    cols = ratings_df['item_id'].map(item_mapping)
    data = np.ones(len(ratings_df))
    user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_mapping), len(item_mapping)))
    return user_item_matrix, user_mapping, item_mapping

#################################
# COMPREHENSIVE EVALUATION
#################################

def comprehensive_evaluation_multiple_rerankers(k=10, sample_size=None):
    print("=" * 80)
    print(f"COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k={k})")
    print("=" * 80)
    
    print("\nLoading MovieLens 100K dataset...")
    ratings_df, movie_df = load_movielens_100k()
    
    print("Splitting data for evaluation...")
    train_df, test_df = train_test_split(
        ratings_df, test_size=0.2, stratify=ratings_df['user_id'], random_state=42
    )
    
    print("Creating user-item matrix...")
    user_item_matrix, user_mapping, item_mapping = create_user_item_matrix(train_df)
    
    reverse_user_mapping = {v: k for k, v in user_mapping.items()}
    reverse_item_mapping = {v: k for k, v in item_mapping.items()}
    
    test_relevant_items = defaultdict(list)
    test_relevant_scores = defaultdict(list)
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        rating = row['rating']
        if user_id in user_mapping and item_id in item_mapping:
            test_relevant_items[user_id].append(item_id)
            test_relevant_scores[user_id].append(rating)
    
    print("\nInitializing Random recommender...")
    model = RandomRecommender(random_state=42)
    model.fit(user_item_matrix)
    
    # Initialisiere Rerankers: Es werden Original Random und unser neuer LLM Reranker verwendet.
    print("\nInitializing rerankers...")
    # LLM initialisieren
    llm = Llama(
        model_path="/home/stef/projects/myenv/models/mistral/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=0,
        verbose=False
    )
    
    # Item-Metadaten laden
    item_metadata = load_item_metadata()
    
    # LLM-Reranker initialisieren
    llm_reranker = LLMReranker(
        model=model,
        llm=llm,
        item_metadata=item_metadata,
        reverse_item_mapping=reverse_item_mapping
    )
    
    # Reranker-Dictionary
    rerankers = {
        "Original Random": None,
        "LLM Reranker": llm_reranker
    }

    
    all_results = {}
    if sample_size is not None and sample_size < len(test_relevant_items):
        eval_users = random.sample(list(test_relevant_items.keys()), sample_size)
    else:
        eval_users = list(test_relevant_items.keys())
    
    print(f"\nEvaluating {len(eval_users)} users...")
    for reranker_name, reranker in rerankers.items():
        print(f"\nEvaluating {reranker_name}...")
        ndcg_scores = []
        precision_scores = []
        recall_scores = []
        all_recs = []
        
        for user_id in eval_users:
            if not test_relevant_items[user_id]:
                continue
            user_idx = user_mapping[user_id]
            if reranker is None:  # Original Random recommender
                rec_idx = model.recommend(user_idx, n=k)
            else:  # Use LLM Reranker
                rec_idx = reranker.rerank(user_idx, n=k)
            rec = [reverse_item_mapping[idx] for idx in rec_idx]
            all_recs.extend(rec_idx)
            ndcg_scores.append(calculate_ndcg(rec, test_relevant_items[user_id], test_relevant_scores[user_id]))
            precision_scores.append(calculate_precision(rec, test_relevant_items[user_id]))
            recall_scores.append(calculate_recall(rec, test_relevant_items[user_id]))
        
        accuracy_metrics = {
            f'ndcg@{k}': np.mean(ndcg_scores),
            f'precision@{k}': np.mean(precision_scores),
            f'recall@{k}': np.mean(recall_scores)
        }
        
        item_popularity = np.zeros(model.n_items)
        for user in range(model.n_users):
            if user in model.user_items:
                for item in model.user_items[user]:
                    item_popularity[item] += 1
        
        diversity_metrics, _ = calculate_diversity_metrics(
            recommendations=all_recs,
            item_popularity=item_popularity,
            total_items=model.n_items
        )
        
        all_results[reranker_name] = {
            'accuracy': accuracy_metrics,
            'diversity': diversity_metrics
        }
    
    print("\n" + "=" * 30 + " ACCURACY METRICS COMPARISON " + "=" * 30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in [f'ndcg@{k}', f'precision@{k}', f'recall@{k}']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Random"]['accuracy'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['accuracy'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Random":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "=" * 30 + " DIVERSITY METRICS COMPARISON " + "=" * 30)
    print(f"{'Metric':<15}", end='')
    for reranker_name in rerankers.keys():
        print(f"{reranker_name:<20}", end='')
    print()
    print("-" * 80)
    for metric in ['item_coverage', 'gini_index', 'shannon_entropy', 'tail_percentage']:
        print(f"{metric:<15}", end='')
        baseline = all_results["Original Random"]['diversity'][metric]
        for reranker_name in rerankers.keys():
            value = all_results[reranker_name]['diversity'][metric]
            change = ((value - baseline) / baseline * 100) if baseline > 0 else float('inf')
            if reranker_name == "Original Random":
                print(f"{value:.4f}{' '*15}", end='')
            else:
                print(f"{value:.4f} ({change:+.1f}%){' '*5}", end='')
        print()
    
    print("\n" + "=" * 30 + " METRIC INTERPRETATIONS " + "=" * 30)
    print("Accuracy Metrics:")
    print("- NDCG: Higher is better, measures ranking quality")
    print("- Precision: Higher is better, measures relevant item ratio in recommendations")
    print("- Recall: Higher is better, measures coverage of relevant items")
    
    print("\nDiversity Metrics:")
    print("- Item Coverage: Higher means more catalog items are recommended")
    print("- Gini Index: Lower means more equality in recommendations")
    print("- Shannon Entropy: Higher means more diverse recommendations")
    print("- Tail Percentage: Higher means more niche items are recommended")
    
    return all_results

if __name__ == "__main__":
    comprehensive_evaluation_multiple_rerankers(k=10)

COMPREHENSIVE EVALUATION WITH MULTIPLE RERANKERS (k=10)

Loading MovieLens 100K dataset...
Splitting data for evaluation...
Creating user-item matrix...


llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



Initializing Random recommender...
Random recommender ready! Total items: 1656

Initializing rerankers...

Evaluating 943 users...

Evaluating Original Random...

Evaluating LLM Reranker...


/home/stef/projects/myenv/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



============================== ACCURACY METRICS COMPARISON ==============================
Metric         Original Random     LLM Reranker        
--------------------------------------------------------------------------------
ndcg@10        0.0087               0.0081 (-7.2%)     
precision@10   0.0139               0.0150 (+7.6%)     
recall@10      0.0060               0.0068 (+13.9%)     

============================== DIVERSITY METRICS COMPARISON ==============================
Metric         Original Random     LLM Reranker        
--------------------------------------------------------------------------------
item_coverage  0.9921               0.9958 (+0.4%)     
gini_index     0.2376               0.2316 (-2.5%)     
shannon_entropy0.9864               0.9875 (+0.1%)     
tail_percentage0.2131               0.2069 (-2.9%)     

============================== METRIC INTERPRETATIONS ==============================
Accuracy Metrics:
- NDCG: Higher is better, measures ranking qua